# 状態管理とセッション

`FileSessionManager`でセッション管理を実装します。このノートブックでは、会話コンテキストを維持し、永続的なエージェント対話のために複数の同時セッションを処理する方法を説明します。

## 学習内容

- 会話状態の永続化を実装する
- 複数のユーザーセッションを同時に管理する
- エージェント再起動をまたいだコンテキストの継続性を処理する
- セッションストレージと取得を設定する

## 前提条件

- [ノートブック03: MCP統合](03-mcp-integration.ipynb)を完了していること
- ファイルシステム操作の理解
- 会話管理の概念に精通していること

📚 **詳細**: [セッション管理ドキュメント](https://strandsagents.com/latest/user-guide/concepts/agents/session-management/)

In [ ]:
import boto3
from strands import Agent
from strands.models import BedrockModel
from strands.session.file_session_manager import FileSessionManager
from strands_tools import image_reader, file_read
from video_reader_local import video_reader_local

print("✅ Imports successful!")

## 会話履歴の理解

デフォルトでは、エージェントは**単一の実行内**で会話履歴を維持します。これは次のことを意味します：
- ✅ エージェントの存続期間中はコンテキストが保持される
- ❌ プログラム終了時にコンテキストが失われる
- ❌ 再起動をまたいだ永続化はない

実際に見てみましょう：

📚 **詳細**: [会話管理](https://strandsagents.com/latest/user-guide/concepts/agents/conversation-management/)

In [ ]:
# Bedrockモデルのセットアップ
session = boto3.Session(region_name='us-west-2')
bedrock_model = BedrockModel(
    model_id="us.anthropic.claude-3-5-sonnet-20241022-v2:0",
    boto_session=session
)

# マルチモーダルエージェントの作成（ノートブック02と03と同じ）
multimodal_agent = Agent(
    model=bedrock_model,
    tools=[image_reader, file_read, video_reader_local],
    system_prompt="""あなたは次のことができるマルチモーダルアシスタントです：
    - 画像の読み取りと分析
    - ドキュメントの処理（PDF、CSV、DOCXなど）
    - 動画の分析と詳細な洞察の提供
    """
)

print("✅ マルチモーダルエージェントを作成しました！")
print("🎨 ツール: image_reader, file_read, video_reader_local")

In [ ]:
# 最初の対話 - 画像を分析
response1 = multimodal_agent("data-sample/diagram.jpgの画像を分析して、何が見えるか教えてください。")
print("応答1:", response1)
print("\n" + "="*80 + "\n")

# 2回目の対話 - エージェントは前の画像分析を覚えている
response2 = multimodal_agent("その画像でどのAWSサービスが見えましたか？")
print("応答2:", response2)
print("\n💡 エージェントは前のメッセージからの画像分析を覚えていました！")

## 会話マネージャー: コンテキストウィンドウの管理

セッションに入る前に、**会話マネージャー**を理解しましょう。

### 課題

会話が長くなるにつれて、次の問題に直面します：
- 🚫 **トークン制限**: モデルには固定のコンテキストウィンドウがある
- ⚡ **パフォーマンス**: コンテキストが大きいほど処理が遅くなる
- 📉 **関連性**: 古いメッセージは関連性が低くなる可能性がある

### 解決策: 会話マネージャー

Strandsは3つの組み込み戦略を提供します：

1. **NullConversationManager**: 管理なし（すべてのメッセージを保持）
2. **SlidingWindowConversationManager**: 最新のN件のメッセージを保持（デフォルト）
3. **SummarizingConversationManager**: 古いメッセージを要約

📚 **詳細**: [会話管理](https://strandsagents.com/latest/user-guide/concepts/agents/conversation-management/)

In [ ]:
from strands.agent.conversation_manager import SlidingWindowConversationManager

# スライディングウィンドウ付きマルチモーダルエージェントの作成（最新10件のメッセージを保持）
windowed_agent = Agent(
    model=bedrock_model,
    tools=[image_reader, file_read, video_reader_local],
    conversation_manager=SlidingWindowConversationManager(
        window_size=10,  # 最新10件のメッセージのみ保持
        should_truncate_results=True  # 大きなツール結果を切り詰める
    ),
    system_prompt="""あなたはスライディングウィンドウメモリを持つマルチモーダルアシスタントで、次のことができます：
    - 画像と動画の分析
    - ドキュメントの処理
    """
)

print("✅ スライディングウィンドウ付きマルチモーダルエージェントを作成しました！")
print("📊 ウィンドウサイズ: 10メッセージ")

In [ ]:
# マルチモーダルコンテンツを含む長い会話をシミュレート
print("スライディングウィンドウをテストするために15件のメッセージを送信中...\n")

for i in range(15):
    if i == 0:
        windowed_agent("data-sample/diagram.jpgの画像を分析してください")
    elif i == 5:
        windowed_agent("data-sample/Welcome-Strands-Agents-SDK.pdfのドキュメントを読んでください")
    else:
        windowed_agent(f"メッセージ {i+1}: AIエージェントについての事実を教えてください")

print(f"\n📊 送信したメッセージの合計: 15")
print(f"📊 メモリ内のメッセージ数: {len(windowed_agent.messages)}")
print(f"\n💡 古いメッセージ（画像とドキュメント分析を含む）が自動的に削除されました！")
print(f"   ウィンドウには最新10件のメッセージのみが残っています。")

### 要約会話マネージャー

古いメッセージを破棄する代わりに、**要約**することができます：

In [ ]:
from strands.agent.conversation_manager import SummarizingConversationManager

# 要約機能付きマルチモーダルエージェントの作成
summarizing_agent = Agent(
    model=bedrock_model,
    tools=[image_reader, file_read, video_reader_local],
    conversation_manager=SummarizingConversationManager(
        summary_ratio=0.3,  # 必要に応じてメッセージの30%を要約
        preserve_recent_messages=5  # 常に最新5件のメッセージを保持
    ),
    system_prompt="""あなたは要約メモリを持つマルチモーダルアシスタントで、次のことができます：
    - 画像と動画の分析
    - ドキュメントの処理
    重要な情報を保持するために、古い会話は要約されます。
    """
)

print("✅ 要約機能付きマルチモーダルエージェントを作成しました！")
print("📝 古いメッセージ（マルチモーダルコンテンツを含む）は破棄されずに要約されます")

## セッション管理: 実行をまたいだ永続化

会話マネージャーは**メモリ内**のコンテキストを処理します。実行をまたいだ**永続的な**会話には、**セッションマネージャー**を使用します。

### 何が永続化されるか？

セッションマネージャーは自動的に以下を保存します：
- 💬 **会話履歴**: すべてのメッセージ
- 🔑 **エージェント状態**: キー値ストレージ
- 📊 **会話マネージャー状態**: 要約、ウィンドウ状態
- 🔄 **マルチエージェント状態**: オーケストレーター設定（Graph/Swarm）

📚 **詳細**: [セッション管理](https://strandsagents.com/latest/user-guide/concepts/agents/session-management/)

In [ ]:
# セッションマネージャーの作成
session_manager = FileSessionManager(
    session_id="multimodal-user-session",
    storage_dir="sessions"
)

# セッションマネージャー付きマルチモーダルエージェントの作成
persistent_multimodal_agent = Agent(
    model=bedrock_model,
    tools=[image_reader, file_read, video_reader_local],
    session_manager=session_manager,
    system_prompt="""あなたは永続的なメモリを持つマルチモーダルアシスタントで、次のことができます：
    - 画像と動画の分析
    - ドキュメントの処理（PDF、CSV、DOCXなど）
    - セッションをまたいだすべての対話を記憶
    """
)

print("✅ セッションマネージャー付きマルチモーダルエージェントを作成しました！")
print(f"📁 セッション保存先: sessions/session_multimodal-user-session/")
print(f"🎨 ツール: image_reader, file_read, video_reader_local")

In [ ]:
# 最初の会話 - 画像を分析
print("=== 📸 最初の対話: 画像分析 ===")
response = persistent_multimodal_agent("data-sample/diagram.jpgの画像を分析して、アーキテクチャを説明してください。")
print(response)
print("\n💾 この対話はセッションに保存されました！")

In [ ]:
# 会話を続ける - エージェントは画像を覚えている
print("\n=== 🔄 2回目の対話: 以前の分析を思い出す ===")
response = persistent_multimodal_agent("そのアーキテクチャ図でどのAWSサービスを特定しましたか？")
print(response)
print("\n✅ エージェントは前の対話からの画像分析を覚えていました！")

In [ ]:
# セッションに動画分析を追加
print("\n=== 🎥 3回目の対話: 動画分析 ===")
response = persistent_multimodal_agent("data-sample/climbing-video.mp4の動画を分析してください")
print(response)
print("\n💾 動画分析もセッションに保存されました！")

## セッションストレージ構造

FileSessionManagerはセッションを構造化された形式で保存します：

```
sessions/
└── session_user-alice-session/
    ├── session.json
    └── agents/
        └── agent_default/
            ├── agent.json
            └── messages/
                ├── message_0.json
                ├── message_1.json
                └── message_2.json
```

## 複数セッションの管理

異なるユーザーやコンテキストに対して異なるセッションを作成できます：

In [ ]:
# ユーザー1のセッション - 画像アナリスト
session_user1 = FileSessionManager(
    session_id="image-analyst-session",
    storage_dir="sessions"
)

agent_user1 = Agent(
    model=bedrock_model,
    tools=[image_reader, file_read, video_reader_local],
    session_manager=session_user1,
    system_prompt="あなたは画像分析を専門とするマルチモーダルアシスタントです。"
)

# ユーザー2のセッション - 動画アナリスト
session_user2 = FileSessionManager(
    session_id="video-analyst-session",
    storage_dir="sessions"
)

agent_user2 = Agent(
    model=bedrock_model,
    tools=[image_reader, file_read, video_reader_local],
    session_manager=session_user2,
    system_prompt="あなたは動画分析を専門とするマルチモーダルアシスタントです。"
)

print("✅ 複数のマルチモーダルセッションエージェントを作成しました！")
print("👤 ユーザー1: 画像アナリスト")
print("👤 ユーザー2: 動画アナリスト")

In [ ]:
# ユーザー1の会話 - 画像分析
print("=== 👤 ユーザー1: 画像アナリスト ===")
response1 = agent_user1("data-sample/diagram.jpgのアーキテクチャ図を分析してください")
print("ユーザー1:", response1)
print("\n" + "="*80 + "\n")

# ユーザー2の会話 - 動画分析
print("=== 👤 ユーザー2: 動画アナリスト ===")
response2 = agent_user2("data-sample/moderation-video.mp4の動画を分析してください")
print("ユーザー2:", response2)
print("\n💡 各ユーザーはマルチモーダルコンテンツを持つ独自の独立したセッションを持っています！")

In [ ]:
# セッションが保存されていることを確認
import os
import json

print("\n" + "="*80)
print("=== 💾 会話が保存されていることを確認 ===")
print("="*80 + "\n")

# ユーザー1のセッションを確認
user1_session_path = "sessions/session_image-analyst-session"
if os.path.exists(user1_session_path):
    print("✅ ユーザー1（画像アナリスト）のセッションが見つかりました:")
    print(f"   📁 パス: {user1_session_path}")
    
    # ユーザー1のメッセージ数をカウント
    user1_messages_dir = os.path.join(user1_session_path, "agents/agent_default/messages")
    if os.path.exists(user1_messages_dir):
        user1_message_count = len([f for f in os.listdir(user1_messages_dir) if f.endswith('.json')])
        print(f"   💬 保存されたメッセージの合計: {user1_message_count}")
    print()
else:
    print("❌ ユーザー1のセッションが見つかりませんでした\n")

# ユーザー2のセッションを確認
user2_session_path = "sessions/session_video-analyst-session"
if os.path.exists(user2_session_path):
    print("✅ ユーザー2（動画アナリスト）のセッションが見つかりました:")
    print(f"   📁 パス: {user2_session_path}")
    
    # ユーザー2のメッセージ数をカウント
    user2_messages_dir = os.path.join(user2_session_path, "agents/agent_default/messages")
    if os.path.exists(user2_messages_dir):
        user2_message_count = len([f for f in os.listdir(user2_messages_dir) if f.endswith('.json')])
        print(f"   💬 保存されたメッセージの合計: {user2_message_count}")

    print()
else:
    print("❌ ユーザー2のセッションが見つかりませんでした\n")


## 組み込みセッションマネージャー

Strands Agentsは2つの本番環境対応セッションマネージャーを提供します：

### 1. FileSessionManager 📁
- **ストレージ**: ローカルファイルシステム
- **使用例**: 開発、テスト、単一マシンでのデプロイ
- **利点**: シンプル、高速、外部依存なし
- **欠点**: 分散システムには適さない

### 2. S3SessionManager ☁️
- **ストレージ**: Amazon S3
- **使用例**: 本番環境、分散システム、クラウドデプロイ
- **利点**: スケーラブル、耐久性、どこからでもアクセス可能
- **欠点**: AWS認証情報とS3バケットが必要

📚 **APIリファレンス**: 
- [FileSessionManager](https://strandsagents.com/latest/api-reference/session/#strands.session.file_session_manager.FileSessionManager)
- [S3SessionManager](https://strandsagents.com/latest/api-reference/session/#strands.session.s3_session_manager.S3SessionManager)

In [ ]:
# セッションデータを検査
import json
import os

session_path = "sessions/session_multimodal-user-session/session.json"
if os.path.exists(session_path):
    with open(session_path, 'r') as f:
        session_data = json.load(f)
    print("📊 セッションデータ:")
    print(json.dumps(session_data, indent=2))
    
    # メッセージ数をカウント
    messages_dir = "sessions/session_multimodal-user-session/agents/agent_default/messages"
    if os.path.exists(messages_dir):
        message_count = len([f for f in os.listdir(messages_dir) if f.endswith('.json')])
        print(f"\n💬 永続化されたメッセージの合計: {message_count}")
        print(f"📦 これにはマルチモーダルコンテンツ（画像、動画、ドキュメント）が含まれます")
else:
    print("ℹ️ セッションファイルはまだ見つかりませんでした。まず永続エージェントのセルを実行してください。")

## S3SessionManager: クラウドネイティブな永続化

本番環境へのデプロイ、特に分散環境では、**S3SessionManager**を使用します。

### 利点:
- ☁️ **クラウドネイティブ**: AWSによって完全に管理される
- 🌍 **分散**: 任意のAWSリージョンからアクセス可能
- 💪 **耐久性**: 99.999999999%の耐久性
- 📈 **スケーラブル**: 数百万のセッションを処理
- 🔒 **セキュア**: IAMベースのアクセス制御

### 必要なS3権限:
```json
{
  "Version": "2012-10-17",
  "Statement": [{
    "Effect": "Allow",
    "Action": [
      "s3:PutObject",
      "s3:GetObject",
      "s3:DeleteObject",
      "s3:ListBucket"
    ],
    "Resource": [
      "arn:aws:s3:::my-sessions-bucket/*",
      "arn:aws:s3:::my-sessions-bucket"
    ]
  }]
}
```

📚 **詳細**: [S3SessionManagerドキュメント](https://strandsagents.com/latest/user-guide/concepts/agents/session-management/#s3sessionmanager)

In [ ]:
from strands.session.s3_session_manager import S3SessionManager

# 設定
BUCKET_NAME = "strands-agents-sessions"  # バケット名を変更してください
SESSION_PREFIX = "production/"  # 組織化のためのオプションのプレフィックス

# S3セッションマネージャーの作成
s3_session_manager = S3SessionManager(
    session_id="user-bob-session",
    bucket=BUCKET_NAME,
    prefix=SESSION_PREFIX,
    boto_session=session,  # boto3セッションを再利用
    region_name="us-west-2"
)

print("✅ S3セッションマネージャーを作成しました！")
print(f"📦 バケット: {BUCKET_NAME}")
print(f"📁 プレフィックス: {SESSION_PREFIX}")


In [ ]:
# S3セッションマネージャー付きマルチモーダルエージェントの作成
s3_multimodal_agent = Agent(
    model=bedrock_model,
    tools=[image_reader, file_read, video_reader_local],
    session_manager=s3_session_manager,
    system_prompt="""あなたはS3バックエンドの永続化を持つクラウドネイティブなマルチモーダルアシスタントで、次のことができます：
    - 画像と動画の分析
    - ドキュメントの処理
    - 本番環境のワークロードにスケール
    """
)

print("✅ S3セッションマネージャー付きマルチモーダルエージェントを作成しました！")

In [ ]:
# マルチモーダルコンテンツでS3セッション永続化をテスト
print("=== 📸 S3永続化で画像を分析 ===")
response = s3_multimodal_agent("data-sample/diagram.jpgのアーキテクチャを分析してください")
print("応答:", response)
print(f"📍 場所: s3://{BUCKET_NAME}/{SESSION_PREFIX}session_user-bob-session/")


### S3ストレージ構造

S3内のセッションはFileSessionManagerと同じ構造に従います：

```
s3://my-sessions-bucket/
└── production/
    └── session_user-bob-session/
        ├── session.json
        └── agents/
            └── agent_default/
                ├── agent.json
                └── messages/
                    ├── message_0.json
                    ├── message_1.json
                    └── message_2.json
```

### S3SessionManagerをいつ使用するか？

✅ **S3SessionManagerを使用する場合:**
- AWSへのデプロイ（Lambda、ECS、EC2）
- 分散システムの構築
- 高い耐久性と可用性が必要
- 複数のサービスがセッションにアクセスする必要がある
- 多くのユーザーにスケールする

❌ **FileSessionManagerを使用する場合:**
- ローカル開発とテスト
- 単一マシンでのデプロイ
- プロトタイピング
- AWSインフラストラクチャがない

## セッション管理の利点

### 1. 継続性
会話は複数の対話と時間をまたぐことができます

### 2. パーソナライゼーション
ユーザーの好みとコンテキストを記憶

### 3. デバッグ
トラブルシューティングのために会話履歴を検査

### 4. 分析
会話パターンとユーザー行動を分析

### 5. 回復
中断後に会話を再開

## 実践例: カスタマーサポートボット

セッション管理付きのカスタマーサポートボットを作成しましょう：

In [ ]:
def create_multimodal_support_agent(customer_id: str):
    """永続セッション付きマルチモーダルカスタマーサポートエージェントを作成します。"""
    session_manager = FileSessionManager(
        session_id=f"customer-{customer_id}",
        storage_dir="sessions"
    )
    
    return Agent(
        model=bedrock_model,
        tools=[image_reader, file_read, video_reader_local],
        session_manager=session_manager,
        system_prompt="""あなたは次のことができるマルチモーダルカスタマーサポートエージェントです：
        - エラーのスクリーンショットを分析
        - ログファイルとドキュメントを確認
        - 問題の動画記録を視聴
        顧客の問題を記憶し、一貫したサポートを提供します。
        親切でプロフェッショナルに接してください。"""
    )

# 顧客用のマルチモーダルサポートエージェントを作成
support_agent = create_multimodal_support_agent("12345")

print("✅ マルチモーダルカスタマーサポートエージェントを作成しました！")
print("🎨 分析可能: スクリーンショット、ドキュメント、動画")

In [ ]:
# 最初のサポート対話 - 顧客がスクリーンショットを共有
print("=== 📸 カスタマーサポート: スクリーンショット分析 ===")
response = support_agent("エラーが表示されています。スクリーンショットです: data-sample/diagram.jpg")
print("サポート:", response[:300] + "...")
print("\n" + "="*80 + "\n")

# フォローアップ - エージェントはスクリーンショットを覚えている
print("=== 🔄 フォローアップ: 以前のコンテキストを記憶 ===")
response = support_agent("スクリーンショット内のAWSサービスが何をしているか説明できますか？")
print("サポート:", response[:300] + "...")
print("\n✅ エージェントはスクリーンショット分析を覚えており、コンテキストを考慮したサポートを提供しました！")

## 比較: FileSessionManager vs S3SessionManager

| 機能 | FileSessionManager | S3SessionManager |
|---------|-------------------|------------------|
| **ストレージ** | ローカルファイルシステム | Amazon S3 |
| **セットアップ** | 設定不要 | S3バケット + IAMが必要 |
| **パフォーマンス** | 非常に高速 | 高速（ネットワークレイテンシあり） |
| **耐久性** | 単一マシン | 99.999999999% |
| **スケーラビリティ** | ディスク容量に制限 | 無制限 |
| **分散** | ❌ いいえ | ✅ はい |
| **コスト** | 無料 | S3ストレージ + APIコスト |
| **使用例** | 開発 | 本番環境 |

### ベストプラクティス

1. **一意のセッションID**: ユーザー/会話ごとに一意のIDを使用
2. **セッションクリーンアップ**: 古いセッションにTTLを実装
3. **エラーハンドリング**: ストレージ障害を適切に処理
4. **監視**: セッションサイズとアクセスパターンを追跡
5. **セキュリティ**: ハードコードされた認証情報ではなく、IAMロールを使用

In [ ]:
!pip install strands-agents strands-agents-tools boto3 -q

## まとめ

このノートブックでは、以下を学習しました：

✅ **会話マネージャー**: コンテキストウィンドウの管理（スライディングウィンドウ、要約）

✅ **セッション管理**: 実行をまたいだ状態の永続化

✅ **FileSessionManager**: ローカルファイルベースの永続化

✅ **S3SessionManager**: クラウドネイティブなS3バックエンドの永続化

✅ **複数セッション**: 異なるユーザー/コンテキストの管理

✅ **ストレージ構造**: セッションの組織化方法

✅ **ベストプラクティス**: 本番環境対応のパターン

### 達成したこと

エージェントは次のことができるようになりました：
- 💬 セッション内の会話を記憶
- 💾 再起動をまたいだ状態の永続化
- 👥 複数のユーザーを独立して処理
- ☁️ S3で本番環境にスケール

### しかし、制限があります...

セッションマネージャーは以下に**優れています**：
- ✅ 会話履歴の維持
- ✅ **この**セッションで言及されたことの記憶
- ✅ 中断された会話の再開

しかし、以下は**できません**：
- ❌ 複数セッション間の検索
- ❌ 過去の会話からの関連情報の検索
- ❌ セッション間のセマンティック関係の理解
- ❌ 過去の対話からの学習

### 質問

**セッション間の理解が必要な場合はどうすればよいでしょうか？**

### 次のステップ

**ノートブック05**に進んで、以下を学習してください：
- エージェント用のAmazon S3 Vectorsのセットアップ
- すべての会話にわたるセマンティック検索の有効化
- 真の長期メモリを持つエージェントの構築
- セッション管理とベクトル検索の組み合わせ

**セッション間の理解を実現しましょう！** 🎯

📚 **リソース**:
- [セッション管理ドキュメント](https://strandsagents.com/latest/user-guide/concepts/agents/session-management/)
- [会話管理](https://strandsagents.com/latest/user-guide/concepts/agents/conversation-management/)
- [APIリファレンス: セッション](https://strandsagents.com/latest/api-reference/session/)